In [ ]:
# ==============================================================================
# TRACK 3 (AVANZADO): PREDICCIÓN Y REABASTECIMIENTO POR CATEGORÍA + REFERENCIA
# ==============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor

# 1. CARGA DE DATOS PROCESADOS
root = Path.cwd()
while not (root / 'data' / 'processed').is_dir():
    root = root.parent
processed_dir = root / 'data' / 'processed'

df_inv = pd.read_csv(processed_dir / 'INVENTARIO_PRODUCTOS_LIMPIO.csv')
df_ventas = pd.read_csv(processed_dir / 'ventas_limpia.csv')

df_inv['codigo'] = df_inv['codigo'].astype(str)
df_ventas['codigo'] = df_ventas['codigo'].astype(str)
df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])

# 2. FILTRAR ESTRICTAMENTE LAS 4 FAMILIAS DE INTERÉS
categorias_analisis = {'LLANTAS', 'BATERIAS', 'ACEITES', 'FILTROS'}
df_inv = df_inv[df_inv['categoria'].isin(categorias_analisis)].copy()
df_ventas = df_ventas[df_ventas['categoria'].isin(categorias_analisis)].copy()

# 3. AGREGACIÓN MENSUAL A NIVEL DE CATEGORÍA + REFERENCIA (CÓDIGO)
df_ventas['periodo'] = df_ventas['fecha'].dt.to_period('M').dt.to_timestamp()

# Agrupamos por periodo, categoría y código (referencia del producto)
ventas_mensuales = df_ventas.groupby(['periodo', 'categoria', 'codigo'])['cantidad'].sum().reset_index()
ventas_mensuales = ventas_mensuales.rename(columns={'cantidad': 'ventas'})

# 4. SIMULACIÓN DE COMPRAS HISTÓRICAS A NIVEL DE REFERENCIA
compras_mensuales = ventas_mensuales.copy()
compras_mensuales['compras'] = compras_mensuales['ventas'] * 1.05

# 5. UNIÓN DE LA HISTORIA COMPLETA
df_historia = pd.merge(
    ventas_mensuales, 
    compras_mensuales[['periodo', 'categoria', 'codigo', 'compras']], 
    on=['periodo', 'categoria', 'codigo'], 
    how='outer'
).fillna(0)

# Ordenar cronológicamente por producto y periodo
df_historia = df_historia.sort_values(by=['codigo', 'periodo']).reset_index(drop=True)

# 6. INGENIERÍA DE CARACTERÍSTICAS (LAGS Y TENDENCIAS POR CADA REFERENCIA)
for lag in [1, 2, 3]:
    df_historia[f'ventas_lag_{lag}'] = df_historia.groupby('codigo')['ventas'].shift(lag)
    df_historia[f'compras_lag_{lag}'] = df_historia.groupby('codigo')['compras'].shift(lag)

df_historia['ventas_media_3m'] = df_historia.groupby('codigo')['ventas'].transform(lambda x: x.rolling(3).mean())

# Eliminar nulos iniciales
df_historia = df_historia.dropna().reset_index(drop=True)

# 7. ENTRENAMIENTO DEL MODELO DE MACHINE LEARNING (RANDOM FOREST)
features = [
    'ventas_lag_1', 'ventas_lag_2', 'ventas_lag_3',
    'compras_lag_1', 'compras_lag_2', 'compras_lag_3',
    'ventas_media_3m'
]
target = 'ventas'

X = df_historia[features]
y = df_historia[target]

modelo_ref = RandomForestRegressor(n_estimators=100, random_state=42)
modelo_ref.fit(X, y)

# 8. DETERMINAR EL PRÓXIMO MES A PREDECIR
# Tomamos el último periodo registrado en los datos y le sumamos 1 mes
ultimo_mes_historico = df_historia['periodo'].max()
proximo_mes = ultimo_mes_historico + pd.DateOffset(months=1)
nombre_mes_proyeccion = proximo_mes.strftime('%Y-%m') # Formato Año-Mes (ej. 2026-11)

# 9. PREDICCIÓN DE LA DEMANDA PARA CADA REFERENCIA EN EL PRÓXIMO MES
ultimos_datos = df_historia.groupby(['categoria', 'codigo']).last().reset_index()
X_futuro = ultimos_datos[features]
ultimos_datos['demanda_pronosticada'] = modelo_ref.predict(X_futuro)

# 10. CRUCE CON EL STOCK REAL ACTUAL Y CÁLCULO DE COMPRAS
stock_real_actual = df_inv[['categoria', 'codigo', 'stock']].drop_duplicates()

plan_final = ultimos_datos[['categoria', 'codigo', 'demanda_pronosticada']].merge(
    stock_real_actual, on=['categoria', 'codigo'], how='left'
).fillna(0)

# Asignar la columna del mes de proyección
plan_final['mes_proyeccion'] = nombre_mes_proyeccion

# Stock de seguridad (20% de la demanda pronosticada)
plan_final['stock_seguridad'] = plan_final['demanda_pronosticada'] * 0.20

# Cálculo de compra sugerida por referencia: Demanda + Seguridad - Stock Real
plan_final['unidades_a_comprar'] = (
    plan_final['demanda_pronosticada'] + plan_final['stock_seguridad'] - plan_final['stock']
)

# Redondeos y formato final
plan_final['unidades_a_comprar'] = np.maximum(0, plan_final['unidades_a_comprar']).round(0)
plan_final['demanda_pronosticada'] = plan_final['demanda_pronosticada'].round(0)
plan_final['stock_seguridad'] = plan_final['stock_seguridad'].round(0)

# Reordenar columnas para legibilidad
plan_final = plan_final[['mes_proyeccion', 'categoria', 'codigo', 'demanda_pronosticada', 'stock', 'stock_seguridad', 'unidades_a_comprar']]

# ==============================================================================
# 11. PRESENTACIÓN EJECUTIVA DE RESULTADOS
# ==============================================================================
print("="*105)
print(f" 🎯 PLAN DE REABASTECIMIENTO POR CATEGORÍA Y REFERENCIA PARA EL MES: {nombre_mes_proyeccion}")
print("="*105)
display(plan_final.head(20)) # Muestra las primeras 20 referencias de ejemplo